# Crypto ETF Event Study and BacktestThis notebook reproduces the ETF milestone event study and backtests a simple spot strategy using BTC data from CoinGecko.

## Setup

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom pathlib import Pathimport etf_analysis as etf

## Load data

In [ ]:
price_path = Path('data/btc_usd_daily.csv')prices = etf.load_price_history(price_path)events = etf.load_events(Path('data/etf_events.csv'))prices.tail(), events

### Price series with ETF milestones

In [ ]:
fig_path = Path('images/btc_price_events.png')etf.plot_event_price(prices, events, fig_path)from IPython.display import ImageImage(filename=str(fig_path))

## Event studyCompute window returns, cumulative curves, and volatility shifts around each event.

In [ ]:
results = etf.compute_event_windows(prices, events, window=30)per_event, summary = etf.summarize_event_results(results)per_event

In [ ]:
summary

In [ ]:
tests = etf.compute_t_tests(per_event)tests

### Cumulative return and drawdown around each event

In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(12, 3 * len(results)), sharex=True)if len(results) == 1:    axes = np.array([axes])for idx, res in enumerate(results):    axes[idx, 0].plot(res.cumulative_curve.index, res.cumulative_curve.values)    axes[idx, 0].axvline(0, color='red', linestyle='--', alpha=0.7)    axes[idx, 0].set_title(f"{res.event_type.title()} - {res.event_date.date()}")    axes[idx, 0].set_ylabel('Cumulative return')    axes[idx, 0].grid(True, linestyle='--', alpha=0.3)    axes[idx, 1].plot(res.drawdown_curve.index, res.drawdown_curve.values, color='orange')    axes[idx, 1].axvline(0, color='red', linestyle='--', alpha=0.7)    axes[idx, 1].set_ylabel('Drawdown')    axes[idx, 1].grid(True, linestyle='--', alpha=0.3)fig.tight_layout()plt.show()

## Backtests- Strategy S1: enter on event close, hold for H days (H=7 and H=30)- Strategy S2: enter 7 days before event, exit on event close, tested on approval/launch events.Transaction costs: 10 bps per side.

In [ ]:
# Strategy S1 - 7 day holdtrades_s1_7, rets_s1_7 = etf.backtest_strategy(prices, events, horizon=7)stats_s1_7 = etf.performance_summary(trades_s1_7, rets_s1_7)# Strategy S1 - 30 day holdtrades_s1_30, rets_s1_30 = etf.backtest_strategy(prices, events, horizon=30)stats_s1_30 = etf.performance_summary(trades_s1_30, rets_s1_30)# Strategy S2 - rumor/sell-the-news for approval+launchapproval_launch = events[events['event_type'].isin(['approval', 'launch'])]trades_s2, rets_s2 = etf.backtest_strategy(prices, approval_launch, horizon=7, entry_shift=-7)stats_s2 = etf.performance_summary(trades_s2, rets_s2)stats_s1_7, stats_s1_30, stats_s2

In [ ]:
# Trade tablesetf.describe_trades(trades_s1_7)

In [ ]:
etf.describe_trades(trades_s1_30)

In [ ]:
etf.describe_trades(trades_s2)

### Equity curves

In [ ]:
etf.plot_equity(rets_s1_7, Path('images/s1_7_equity.png'), 'Strategy S1 - 7 day hold')etf.plot_equity(rets_s1_30, Path('images/s1_30_equity.png'), 'Strategy S1 - 30 day hold')etf.plot_equity(rets_s2, Path('images/s2_equity.png'), 'Strategy S2 - pre-event entry')from IPython.display import Image, displayfor path in ['images/s1_7_equity.png', 'images/s1_30_equity.png', 'images/s2_equity.png']:    display(Image(filename=path))